## NB-03 — Document Ingestion and Chunking

The quality of every downstream retrieval decision depends on how documents are
split into chunks. This notebook teaches you to load ShopSmart India's mixed
knowledge base — structured product CSVs, Markdown policy documents, and HTML
buyer FAQs — and apply the right chunking strategy to each file type.
The central insight is that a single product CSV row is already a natural semantic
unit (it contains all facts about one product), while unstructured policy prose
must be split by sentence boundaries to avoid cutting a policy rule in half.

### Concepts Covered

- Writing LangChain document loaders for CSV, Markdown, and HTML file types
- Row-level chunking for product catalogue CSV: why one chunk per product is better than grouping rows
- Fixed-size chunking (512 tokens, 50 overlap) vs. recursive sentence splitting for policy documents
- Metadata tagging: `source_file`, `file_type`, `product_id`, `section_heading`, `question_number`
- Inspecting chunk quality: does a product chunk contain enough context to answer a spec question?
- Chunk statistics: count, average length, and token distribution per file type
- Understanding how chunking strategy affects NB-04 retrieval quality

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import csv
import json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from bs4 import BeautifulSoup
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain.schema import Document

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Paths — create output dir if absent
DATA_DIR = Path("data")
os.makedirs("output", exist_ok=True)

print("Document loaders and chunkers imported.")
print(f"Looking for knowledge base files in: {DATA_DIR.resolve()}")


### Step 1 — CSV Product Catalogue Loader (Row-Level Chunking)

Each row in `products.csv` is a single product record. We chunk at row level:
one `Document` per product. This is the right strategy because:

1. A shopper's product question is answered entirely within one row — we never
   need to aggregate multiple rows to answer "what is the battery life of boAt Airdopes 141?"
2. Grouping rows creates chunks that are too long and mix product facts from
   different products, confusing the retrieval model.
3. Each chunk gets `product_id` in metadata, enabling citation and metadata filtering.

In [ ]:
# ── CSV ROW-LEVEL LOADER ─────────────────────────────────────

def load_product_catalogue(csv_path: Path) -> list[Document]:
    """
    Load products.csv as one Document per row.
    Each document's page_content is a human-readable text representation
    of all product fields. Metadata contains product_id for citation.
    """
    docs = []
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from {csv_path.name}")
    print(f"Columns: {list(df.columns)}")

    for _, row in df.iterrows():
        # Build a human-readable text block for this product
        # This is what the embedding model will encode
        text_parts = [
            f"Product: {row.get('name', 'Unknown')}",
            f"Brand: {row.get('brand', 'Unknown')}",
            f"Category: {row.get('category', 'Unknown')} > {row.get('subcategory', '')}",
            f"Price: ₹{row.get('price_inr', 'N/A')}",
            f"Rating: {row.get('rating', 'N/A')}/5 ({row.get('rating_count', 0)} reviews)",
            f"Specs: {row.get('specs', 'N/A')}",
            f"Warranty: {row.get('warranty_months', 0)} months",
            f"Returnable: {'Yes' if row.get('returnable', False) else 'No'} "
            f"({row.get('return_window_days', 0)}-day window)",
            f"Seller ID: {row.get('seller_id', 'Unknown')} "
            f"(Verified: {'Yes' if row.get('seller_verified', False) else 'No'})",
            f"Description: {row.get('description', '')}",
        ]
        page_content = "\n".join(text_parts)

        doc = Document(
            page_content=page_content,
            metadata={
                "source_file": str(csv_path.name),
                "file_type": "csv",
                "product_id": str(row.get("product_id", "UNKNOWN")),
                "product_name": str(row.get("name", "")),
                "category": str(row.get("category", "")),
                "price_inr": float(row.get("price_inr", 0)),
            }
        )
        docs.append(doc)

    return docs

# Load the product catalogue
products_path = DATA_DIR / "products.csv"
if products_path.exists():
    product_docs = load_product_catalogue(products_path)
    print(f"\nCreated {len(product_docs)} product chunks (one per row)")
    print("\n── Sample chunk (first product) ────────────────────────────")
    print(product_docs[0].page_content)
    print(f"\nMetadata: {product_docs[0].metadata}")
else:
    print(f"⚠️  {products_path} not found. Create the file from data/README.md.")
    print("   Using placeholder list for demonstration.")
    product_docs = []


### Step 2 — Markdown Loader with Section-Aware Chunking

Policy documents and the seller guide are Markdown files with section headings.
We use `MarkdownHeaderTextSplitter` to split at `##` and `###` boundaries first,
which preserves policy rule boundaries. We then apply `RecursiveCharacterTextSplitter`
to any resulting chunks that are still too long. Each chunk inherits the section
heading as metadata — this becomes the citation reference in NB-05.

In [ ]:
# ── MARKDOWN LOADER WITH RECURSIVE SPLITTING ─────────────────

def load_markdown_doc(md_path: Path) -> list[Document]:
    """
    Load a Markdown policy or guide document.
    First split on headers (preserves policy section boundaries),
    then apply recursive character splitting on oversized sections.
    """
    text = md_path.read_text(encoding="utf-8")

    # Step 1: split on Markdown headers to keep policy sections intact
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "h1"),
            ("##", "section_heading"),
            ("###", "subsection_heading"),
        ],
        strip_headers=False,    # keep headings inside the chunk for context
    )
    header_docs = header_splitter.split_text(text)

    # Step 2: further split any section that is still too long
    # 512 tokens ≈ 2048 chars for all-MiniLM-L6-v2 (max 256 word-pieces)
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,        # characters (approx 300-400 tokens)
        chunk_overlap=150,
        separators=["\n\n", "\n", ". ", " "],
    )

    final_docs = []
    for hdoc in header_docs:
        if len(hdoc.page_content) > 1500:
            # Split oversized sections further
            sub_docs = char_splitter.create_documents(
                [hdoc.page_content],
                metadatas=[hdoc.metadata]
            )
            final_docs.extend(sub_docs)
        else:
            final_docs.append(hdoc)

    # Add file-level metadata to every chunk
    for doc in final_docs:
        doc.metadata.update({
            "source_file": md_path.name,
            "file_type": "markdown",
        })

    return final_docs

# Load all Markdown documents
md_files = {
    "returns_policy": DATA_DIR / "returns_policy.md",
    "seller_onboarding": DATA_DIR / "seller_onboarding.md",
    "category_taxonomy": DATA_DIR / "category_taxonomy.md",
}

markdown_docs = []
for doc_name, md_path in md_files.items():
    if md_path.exists():
        docs = load_markdown_doc(md_path)
        markdown_docs.extend(docs)
        print(f"{md_path.name}: {len(docs)} chunks")
        if docs:
            print(f"  Sample heading: {docs[0].metadata.get('section_heading', 'N/A')}")
    else:
        print(f"⚠️  {md_path} not found — skipping.")

print(f"\nTotal Markdown chunks: {len(markdown_docs)}")


### Step 3 — HTML FAQ Loader

The buyer FAQ (`buyer_faq.html`) uses `<article>` tags with an `id` attribute
per question. We parse with BeautifulSoup, extract each question-answer pair,
and create one chunk per FAQ item. The `question_number` metadata (e.g. `faq-021`)
lets the citation system point to the exact FAQ item.

In [ ]:
# ── HTML FAQ LOADER ──────────────────────────────────────────

def load_buyer_faq(html_path: Path) -> list[Document]:
    """
    Parse buyer_faq.html and create one Document per FAQ article.
    Preserves the section heading (e.g. 'Returns & Refunds') as metadata
    so filters can target FAQ by topic.
    """
    html_text = html_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html_text, "lxml")

    docs = []
    current_section = "General"  # fallback if no section heading found

    for element in soup.find_all(["h2", "article"]):
        if element.name == "h2":
            # Track the current section heading for metadata
            current_section = element.get_text(strip=True)

        elif element.name == "article":
            question_num = element.get("id", "faq-unknown")
            # Grab all text content from this FAQ item
            raw_text = element.get_text(separator=" ", strip=True)
            # Clean up excessive whitespace
            clean_text = " ".join(raw_text.split())

            doc = Document(
                page_content=clean_text,
                metadata={
                    "source_file": html_path.name,
                    "file_type": "html",
                    "question_number": question_num,
                    "section_heading": current_section,
                }
            )
            docs.append(doc)

    return docs

faq_path = DATA_DIR / "buyer_faq.html"
if faq_path.exists():
    faq_docs = load_buyer_faq(faq_path)
    print(f"buyer_faq.html: {len(faq_docs)} FAQ chunks")
    if faq_docs:
        print("\n── Sample FAQ chunk ────────────────────────────────────────")
        print(faq_docs[0].page_content[:300])
        print(f"\nMetadata: {faq_docs[0].metadata}")
else:
    print(f"⚠️  {faq_path} not found — skipping.")
    faq_docs = []

print(f"\nTotal FAQ chunks: {len(faq_docs)}")


### Step 4 — Chunking Strategy Comparison (Fixed-size vs Recursive)

We test fixed-size chunking against recursive sentence splitting on the returns
policy to understand the tradeoff. Fixed-size chunking is fast but can cut a
policy rule mid-sentence. Recursive splitting respects sentence boundaries but
produces variable-length chunks.

In [ ]:
# ── CHUNKING STRATEGY COMPARISON ────────────────────────────

returns_path = DATA_DIR / "returns_policy.md"

if returns_path.exists():
    raw_text = returns_path.read_text(encoding="utf-8")

    # Strategy A: Fixed-size chunking
    fixed_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,        # characters
        chunk_overlap=50,
        separators=["\n\n", "\n", " "],   # try paragraph, then line, then word
    )
    fixed_chunks = fixed_splitter.create_documents([raw_text])

    # Strategy B: Markdown-header-aware recursive splitting (our chosen strategy)
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
        strip_headers=False,
    )
    recursive_chunks = header_splitter.split_text(raw_text)

    print("── Chunking Strategy Comparison: returns_policy.md ─────────")
    print(f"\nStrategy A — Fixed-size (512 chars, 50 overlap):")
    print(f"  Chunks created: {len(fixed_chunks)}")
    lengths_a = [len(c.page_content) for c in fixed_chunks]
    print(f"  Min/Mean/Max chars: {min(lengths_a)}/{sum(lengths_a)//len(lengths_a)}/{max(lengths_a)}")

    print(f"\nStrategy B — Markdown header-aware:")
    print(f"  Chunks created: {len(recursive_chunks)}")
    lengths_b = [len(c.page_content) for c in recursive_chunks]
    if lengths_b:
        print(f"  Min/Mean/Max chars: {min(lengths_b)}/{sum(lengths_b)//len(lengths_b)}/{max(lengths_b)}")

    print("\n── Inspecting a policy boundary ──────────────────────────────")
    print("Strategy A — chunk that may cut a policy rule mid-sentence:")
    print(repr(fixed_chunks[2].page_content[:200]) if len(fixed_chunks) > 2 else "N/A")
    print("\nStrategy B — same content with section heading preserved:")
    if len(recursive_chunks) > 1:
        print(repr(recursive_chunks[1].page_content[:300]))
else:
    print(f"⚠️  {returns_path} not found — create it from data/README.md template.")


In [ ]:
# ── CHUNK STATISTICS ─────────────────────────────────────────

all_docs = product_docs + markdown_docs + faq_docs

print("── Chunk Statistics by File Type ────────────────────────────")
print(f"{'File Type':<15} {'Count':>7} {'Avg Chars':>10} {'Min':>6} {'Max':>6}")
print("─" * 50)

for file_type, docs in [
    ("csv (products)", product_docs),
    ("markdown",       markdown_docs),
    ("html (faq)",     faq_docs),
]:
    if docs:
        chars = [len(d.page_content) for d in docs]
        print(f"{file_type:<15} {len(docs):>7} {sum(chars)//len(docs):>10} "
              f"{min(chars):>6} {max(chars):>6}")

print(f"\n{'TOTAL':<15} {len(all_docs):>7}")

# Save all docs to a JSON file for NB-04 to load
os.makedirs("output", exist_ok=True)
serialised = [
    {"page_content": d.page_content, "metadata": d.metadata}
    for d in all_docs
]
with open("output/all_chunks.json", "w", encoding="utf-8") as f:
    json.dump(serialised, f, indent=2, ensure_ascii=False)
print(f"\nAll {len(all_docs)} chunks saved to output/all_chunks.json for NB-04.")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Row grouping vs row-level (why single-row chunking is better)
# Modify load_product_catalogue to group 5 rows per chunk instead of 1.
# Then ask: does a chunk for boAt Airdopes 141 now contain battery life info
# clearly, or is it buried among 4 other products? Print the chunk and inspect.
# Run the same question through a simple keyword search on both chunk sets.

# EXERCISE 2 — Chunk size sensitivity on policy documents
# Try fixed_splitter with chunk_size=256 and chunk_size=1024 on returns_policy.md.
# Count how many chunks cut the 'Sale & Promotional Items' section in half.
# At which chunk size does the section land cleanly in one chunk?

# EXERCISE 3 — Metadata-driven chunking for the seller guide
# The seller_onboarding.md has numbered checklist items.
# Write a custom splitter that treats each checklist item (lines starting with '- [ ]')
# as a separate chunk. Add a metadata field 'checklist_item': True.
# This would allow a metadata filter like: filter(checklist_item=True) for compliance queries.

print("Experiments ready. Modify loader functions above and re-run cells.")


### Key Takeaway

Chunking is not a one-size-fits-all operation. Structured product data demands
row-level chunking so that all facts about a single product stay together;
unstructured policy prose demands boundary-aware splitting so that policy rules
are never cut mid-sentence. The metadata tagging strategy — `product_id` for
CSV rows, `section_heading` for Markdown, `question_number` for HTML — enables
accurate citations and metadata filtering in the retrieval layer.

In **NB-04** you will take the `output/all_chunks.json` produced here, generate
embeddings for every chunk, and store them in a FAISS vector index — then run
five test queries, one per CatalogueIQ query type, to inspect raw retrieval
quality before connecting generation.